In [1]:
from nltk.corpus import movie_reviews

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
from sklearn.model_selection import train_test_split
from collections import Counter
import re

In [6]:
# 사용자 정의 토크나이저
class SimpleTokenizer:
    def __init__(self, num_words = 10000, oov_token='UNK'):
        self.num_words = num_words
        self.oov_token = oov_token
        self.word_index = {}
        self.index_word = {}

    def _clean_text(self, text):
        return ' '.join(re.findall(r'\w+', text)) # 또는 re.sub(r'[^\w\s]', '', text).strip()
    
    def fit_on_texts(self, texts):
        '''빈도순으로 상위 단어 추출 토큰을 숫자로 변경, 공백을 기준으로 토큰 분류'''
        word_counts = Counter()

        for text in texts:
            word_counts.update(self._clean_text(text))

            # 빈도순으로 num_words 단어 추출
            most_common = word_counts.most_common(self.num_words-2) # pad UNK 특수 토큰 자리 남기기

            # 0: padding 1: oov
            self.word_index = {self.oov_token: 1}
            for i, (word, _) in enumerate(most_common):
                self.word_index[word] = i + 2
            
            self.index_word = {idx: w for w, idx in self.word_index.items()}

    def texts_to_sequence(self, texts):
        sequence = []

        for text in texts:
            seq = []
            for word in self._clean_text(text):
                seq.append(self.word_index.get(word,1))
            sequence.append(seq)

        return sequence
    
def pad_sequence(sequences, maxlen, padding='pre', truncating='pre'):

    features = np.zeros((len(sequences), maxlen), dtype=int)

    for i, seq in enumerate(sequences):
        if len(seq) > maxlen:
            if truncating == 'pre':
                features[i,:] = seq[-maxlen:]

            else:
                features[i,:] = seq[:maxlen]
        
        else:
            if padding == 'pre':
                features[i,-len(seq):] = seq
            
            else:
                features[i,:len(seq)] = seq
         
    return features

In [5]:
texts = ['i love you', 'i like music', 'love you']
sim = SimpleTokenizer()
sim.texts_to_sequence(texts)

[[1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
 [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
 [1, 1, 1, 1, 1, 1, 1, 1]]

In [7]:
# 리뷰 데이터로 적용해서 오류 없는지 확인 및 수정
reviews = [movie_reviews.raw(fileid) for fileid in movie_reviews.fileids()]

In [8]:
texts = reviews[0:2]
sim = SimpleTokenizer()
sim.fit_on_texts(texts)
requens = sim.texts_to_sequence(texts)

In [9]:
features = pad_sequence(requens,500)

In [10]:
features.shape

(2, 500)

In [17]:
# RNN 모델
class RNNModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Sequential([
            nn.Linear(hidden_dim, 32),
            nn.ReLU(),
            nn.Linear(32,1),
        ])

    def forward(self,x):
        x = self.embedding(x)
        _, hn = self.rnn(x) # output, hn
        # output: 모든 시점(time-step)의 숨겨진 상태: 각 단어(시점)를 거칠때마다 계산된 모든 hidden state를 모아놓음
        # seq2seq 모델은 각 단어마다 결과를 내야하는 개체명 인식(NER)

        # hn: 마지막 시점의 상태; 전체 문장을 다 읽고 최종적으로 요약한 정보: 문서분류
        return self.fc(hn.squeeze(0))
    

In [ ]:
# 테스트
texts = reviews[0:2]
sim = SimpleTokenizer()
sim.fit_on_texts(texts) # 문자 -> 숫자
requens = sim.texts_to_sequence(texts) # 길이를 맞춤
features = pad_sequence(requens,500)
features = torch.LongTensor(features)
print(features.shape)
features = nn.Embedding(500,32)(features)
outputs, hn = nn.RNN(32,64,batch_first=True)(features)
outputs.shape, hn.shape

torch.Size([2, 500])


(torch.Size([2, 500, 64]), torch.Size([1, 2, 64]))

In [ ]:
# 데이터를 가져오기
# x, y 분할
# 토크나이저 + pad_sequence --> 문자를 수자로 변환

# train, test split
# TorchTensor 변환
# TensorDataset --> Dataloader

# 모델 생성
# 옵티마이저
# 손실 함수 정의


